# Post5G - Network Slicing

## Prerequisites
* You have successfully followed the [Simple Scenario](./simple.ipynb) and [TDD](./tdd.ipynb) tutorials.
* You have basic Python knowledge
* You have a good understanding of the 5G architecture
* You have a good understanding of network slicing

## Experiment for this scenario

It the previous tutorials we have configured and deployed 5G networks. In this
tutorial we will configure network slices.

We will deploy a single 5G core with two slices, `internet` and `streaming`. One
UE will use the first slice and a second UE will be on the second slice. The
details of the slices are provided in the table below.

| Slice | SST | SD | DNN | IPv4 Network | Uplink AMBR | Downlink AMBR | QFI | 5QI | ARP Priority | DNNs |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| internet | 1 (eMBB) | `0xFFFFFF` | internet | `12.1.1.0/24` | 20 Mbps | 40 Mbps | 1 | 9 | 8 | {internet} |
| streaming | 1 (eMBB)| `0x100000` | streaming | `14.1.1.0/24` | 30 Mbps | 50 Mbps | 1 | 5 | 1 | {streaming} |

| UE | SUPI | S-NSSAI | DNN |
| --- | --- | --- | --- |
| `sim_ue1` | `001010000000100` | `internet` | `internet` |
| `sim_ue2` | `001010000000101` | `streaming` | `streaming` |


To examplify this use case, each slice will have its own UFP: `upf1` is the UPF
in charge of the `internet` slice while `upf2` is for the `streaming` slice as
shown in the figure below.

![Slicing topology](./figures/slicing.png)

The core network uses Open5GS and the RAN uses OAI. The radio channel between
the UEs and the gNB is simulated with `rfsim`.

## Metamodel and Deployement model definition

### Metamodel
In our previous tutorials, the metamodel remained simple as we leveraged its
default values. In this case as we are adding slices and DNNs, we have to
specify, for each network function, the slices and/or DNNs it refers too.

First we have to define the slices and name them, here we have the first slice
that we call `internet` and the second one that we name `streaming`. For each
function we will reference the slices via their friendly name.

For example, we specify that `upf1` is for the S-NSSAI named `internet` and our
framework will automatically know how to configure the UPF.

Slicing is configured on the UEs by setting their `subscriptions` attribute as
desired.

In [1]:
from metamodel.models.metamodel import MetaModel

metamodel=MetaModel.from_yaml(
"""
slices:
  internet:
    sst: 1
    sd: 0xFFFFFF
    dnns:
      internet:
        ipv4Network: 12.1.1.0/24
        aggregateMaximumBitRate:
          uplink: 20Mbps
          downlink: 40Mbps
        qosFlows:
        - qfi: 1
          fiveQI: 9
          arp:
            priorityLevel: 8
  streaming:
    sst: 1
    sd: 0x100000
    dnns:
      streaming:
        ipv4Network: 14.1.1.0/24
        aggregateMaximumBitRate:
          uplink: 30Mbps
          downlink: 50Mbps
        qosFlows:
        - qfi: 1
          fiveQI: 5
          arp:
            priorityLevel: 1
core:
  components:
    upf1:
      type: upf
      config:
        snssaiList:
          - internet
    upf2:
      type: upf
      config:
        snssaiList:
          - streaming
          
    nrf:
      type: nrf
      config:
        plmnList:
        - plmnId:
            mcc: '001'
            mnc: '01'
          snssaiList:
            - internet
            - streaming

    smf:
      type: smf
      config:
        snssaiList:
          - internet
          - streaming
        plmnList:
        - mcc: '001'
          mnc: '01'
        upfs:
          - upf1
          - upf2

    pcf:
      type: pcf

    nssf:
      type: nssf
      config:
        nsiInfoList:
          - snssai: internet
          - snssai: streaming
        taInfoList:
        - tai:
            plmnId:
              mcc: '001'
              mnc: '01'
            tac: 1
          snssaiList:
            - internet
            - streaming
        amfInfoList:
        - amfList:
          - supportedNssaiAvailabilityData:
              tai:
                plmnId:
                  mcc: '001'
                  mnc: '01'
                tac: 1
              snssaiList:
                - internet
                - streaming
        plmnInfoList:
        - homePlmnId:
            mcc: '001'
            mnc: '01'

    udr:
      type: udr

    amf:
      type: amf
      config:
        guamiList:
          - plmnId:
              mcc: '001'
              mnc: '01'
        taiList:
        - plmnId:
            mcc: '001'
            mnc: '01'
          tac: 1
        plmnList:
        - plmnId:
            mcc: '001'
            mnc: '01'
          snssaiList:
            - internet
            - streaming

    udm:
      type: udm

    bsf:
      type: bsf

    ausf:
      type: ausf
      config:
        plmnList:
          - mcc: '001'
            mnc: '01'

ran:
  components:
    gnb:
      type: gnb
      config:
        plmnList:
          - plmnId:
              mcc: '001'
              mnc: '01'
            snssaiList:
              - internet
              - streaming
        amf: amf
ues:
  sim_ue1:
    config:
      supi: '001010000000100'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - gnb
  sim_ue2:
    config:
      supi: '001010000000101'
      subscriptions:
      - snssai: streaming
        dnn: streaming
      gnbs:
      - gnb
""")

**NOTE:** as the radio channel is simulated we explicitly list the gNBs to which
the UEs are connected.

### Deployment model

There is no specific needs for the deployment model but for the sake of the
example we show how to specify the implementation for each 5G core function.

In [2]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
core:
  components:
    upf:
      type: upf
      implementation: open5gs
    nrf:
      type: nrf
      implementation: open5gs
    smf:
      type: smf
      implementation: open5gs
    pcf:
      type: pcf
      implementation: open5gs
    nssf:
      type: nssf
      implementation: open5gs
    udr:
      type: udr
      implementation: open5gs
    amf:
      type: amf
      implementation: open5gs
    udm:
      type: udm
      implementation: open5gs
    bsf:
      type: bsf
      implementation: open5gs
    ausf:
      type: ausf
      implementation: open5gs

ran:
  components:
    gnb:
      type: gnb
      implementation: oai
ues:
  sim_ue1:
    implementation: oai
    type: containerized
  sim_ue2:
    implementation: oai
    type: containerized
nodes:
  node-core:
    provisioner: my-provisionner
    roles:
    - core
    - ric
  node-ran:
    provisioner: my-provisionner
    roles:
    - ran
  node-ues:
    provisioner: my-provisionner
    roles:
    - ue
provisioners:
  my-provisionner:
    type: bi
    params:
      infra: fr-sophia2-bi-vm1
      project: post5g-beta
      experiment: slicing_scenario
      create_experiment: true
      experiment_duration: 8h
      node_duration: 8h
"""
)



## Provisioning the infrastructure

We provision the infrastructure as in the [Simple tutorial](./simple.ipynb)

In [3]:
from orchestrator.deployment import ProvisioningManagement, K8SManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision(quiet=False)

k8s = K8SManagement(preceding=prov)
await k8s.deploy(quiet=True)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node-ues' provisioning] ***

TASK [provisioning/bi/provision : Select project 'post5g-beta' fo

At this stage, the infrastructure is fully ready to support our 5G network
deployment.

But before deploying the 5G network, let's archive the metamodel and deployment
model resulting from this deployment.

In [4]:
from pathlib import Path
DIR="state/slicing"
Path(DIR).mkdir(parents=True, exist_ok=True)

In [5]:

prov.metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml")
prov.deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml")

k8s.metamodel.to_yaml_file(f"{DIR}/metamodel.k8s.yaml")
k8s.deployment_model.to_yaml_file(f"{DIR}/deployment_model.k8s.yaml")

## Deploy the 5G Network

### Deploy the 5G core

Now that the infrastructure is fully provisioned, we first have to deploy the 5G
core and check that it is composed of two UPFs.

In [6]:
from orchestrator.deployment import CoreManagement
core = CoreManagement(preceding=k8s)
await core.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-pcf' chart] ******

TASK [helm/prepare : Create Helm build directory for 'open5gs-udr' chart] ******

TASK [helm/prepare : Create Helm build directory for 'open5gs-udm' chart] ******

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-upf1' chart] *****

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-upf2' chart] *****

TASK [helm/prepare : Create Helm build directory for 'open5gs-nrf' chart] ******

PLAY [node-core

In [7]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(preceding=core)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "open5gs-core", "-o", "wide"],
    nodes=[prov.deployment_model.params.k8s.master_node],
    become=True, quiet=True))[prov.deployment_model.params.k8s.master_node])

NAME                            READY   STATUS    RESTARTS   AGE     IP               NODE        NOMINATED NODE   READINESS GATES
open5gs-amf-7b6dcfbfdf-8m7kv    1/1     Running   0          2m9s    10.244.134.7     node-core   <none>           <none>
open5gs-ausf-5778d7548b-v45lf   1/1     Running   0          114s    10.244.134.8     node-core   <none>           <none>
open5gs-bsf-75cf9d7fb4-f67gw    1/1     Running   0          20s     10.244.134.14    node-core   <none>           <none>
open5gs-database-0              1/1     Running   0          2m54s   10.244.134.5     node-core   <none>           <none>
open5gs-nrf-65d98494d7-pzzs2    1/1     Running   0          2m29s   10.244.134.6     node-core   <none>           <none>
open5gs-nssf-84cb8d64f7-2q4sl   1/1     Running   0          34s     10.244.134.13    node-core   <none>           <none>
open5gs-pcf-6f444b66c5-xzbj7    1/1     Running   0          83s     10.244.134.10    node-core   <none>           <none>
open5gs-smf-5c7

### Deploy the 5G RAN

We can now deploy the 5G RAN and verify that the gNB is deployed properly and 
supports both slices by analysing its logs.

In [8]:
from orchestrator.deployment import RANManagement
ran = RANManagement(preceding=core)
await ran.deploy()


PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-gnb' chart] **********
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]

TASK [helm/prepare : Copy local 'oai-gnb' Helm chart directory for 'oai-gnb'] ***
changed: [node-core]

TASK [helm/prepare : Render 'oai-gnb' Helm Chart.yaml file from Jinja template] ***
changed: [node-core]

TASK [helm/prepare : Render templated Helm files for 'oai-gnb'] ****************
changed: [node-core] => (item=values.yaml)
changed: [node-core] => (item=config.yaml)

PLAY RECAP *********************************************************************
node-core                  : ok=4

In [9]:
print(await ran.get_logs(components=["gnb"], quiet=True))

/proc/sys/kernel/core_pattern=|/usr/share/apport/apport -p%p -s%s -c%c -d%d -P%P -u%u -g%g -F%F -- %E
No configuration file /opt/oai-gnb/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

Active_gNBs:
  - gnb
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  ##### Identification parameters:
  - gNB_ID: 0x1

    gNB_name: gnb

    tracking_area_code: 1

    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1
            sd: 0xFFFFFF
          - sst: 1
            sd: 0x100000

    nr_cellid: 1

    ##### Physical parameters:
    min_rxtxtime: 6

    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 0
        #  downlinkConfigCommon
        #frequencyInfoDL
        # this is 3300.60 MHz + 53*12*30e-3 MHz = 3319.68
        absoluteFrequencySSB: 621312
        # this is 3300.60 MHz
        dl_absoluteFrequencyPointA: 620040
        #scs-SpecificCarrierList
        

## Deploy the UEs

Now that we have a fully functional core and RAN, we can deploy the two UEs we
have defined in our metamodel. `sim_ue1` is in the slice `internet` and should
then obtain an IP address in the `12.1.1.0/24` prefix upon successful PDU
session establishement. As `sim_ue2` is in the slice `streaming`, it should get
an IP in the `14.1.1.0/24` prefix.


In [10]:
from orchestrator.deployment import UEManagement
ues = UEManagement(preceding=ran)
await ues.deploy()


PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue1' chart] ******

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue2' chart] ******
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]

TASK [helm/prepare : Copy local 'oai-containerized' Helm chart directory for 'oai-sim-ue1'] ***
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.co

In [11]:
command = ["bash", "-c", "'ip -4 addr show oaitun_ue1 | grep -oP \"(?<=inet )\\d+(\\.\\d+){3}\"'"]
ips = (await ues.run_command(command=command, components=["sim_ue1", "sim_ue2"], quiet=True))
for name, ip in ips.items():
    print (f"IP address of {name}: {ip.rstrip()}")

IP address of sim_ue1: 12.1.1.2
IP address of sim_ue2: 14.1.1.2


We can also validate that the UEs use the correct slices by analysing their
logs.

In [12]:
print(await ues.get_logs(components=["sim_ue1"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-c8b78200bf7392b842f7e5b8485837531acaa3e4 Ab

In [13]:
print(await ues.get_logs(components=["sim_ue2"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-c8b78200bf7392b842f7e5b8485837531acaa3e4 Ab

Logs confirm that `sim_ue1` uses the `internet` slice.

```
[SIM]    will request PDU session ID 1, type 1, DNN internet, NSSAI 1.ffffff
```

and that `sim_ue2` uses the `streaming` slice.

```
[SIM]    will request PDU session ID 1, type 1, DNN streaming, NSSAI 1.100000
```

To test that the UEs are well connected and that our network works as expected
we can run a ping from the UEs via their radio interace. In UERANSIM, the
interface is called `uesimtun0`.

To run commands directly on the nodes,

In [14]:
command = ["ping", "-I", "oaitun_ue1", "-c", 10, "-D", "8.8.8.8"]
pings = await ues.run_command(command=command, components=["sim_ue1", "sim_ue2"], quiet=True)

ping from `sim_ue1`:

In [15]:
print (pings["sim_ue1"])

PING 8.8.8.8 (8.8.8.8) from 12.1.1.2 oaitun_ue1: 56(84) bytes of data.
[1791540401.441721] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=26.2 ms
[1791540402.446410] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=29.6 ms
[1791540403.452544] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=34.5 ms
[1791540404.443845] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=24.8 ms
[1791540405.449843] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=29.7 ms
[1791540406.448681] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=26.8 ms
[1791540407.446951] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=23.9 ms
[1791540408.451092] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=27.1 ms
[1791540409.449482] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=24.7 ms
[1791540410.453118] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=26.4 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9010ms
rtt min/avg/max/mdev = 23.937/27.385/34.518/2.994 ms



ping from `sim_ue2`:

In [16]:
print (pings["sim_ue2"])

PING 8.8.8.8 (8.8.8.8) from 14.1.1.2 oaitun_ue1: 56(84) bytes of data.
[1791540401.590817] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=23.0 ms
[1791540402.597732] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=29.5 ms
[1791540403.597528] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=28.4 ms
[1791540404.603625] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=33.6 ms
[1791540405.600116] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=28.7 ms
[1791540406.603712] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=30.5 ms
[1791540407.603002] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=27.4 ms
[1791540408.601838] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=26.8 ms
[1791540409.602222] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=25.8 ms
[1791540410.606202] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=29.1 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9009ms
rtt min/avg/max/mdev = 23.008/28.284/33.614/2.699 ms



Finally, we can archive the metamodel and deployment model in file for keeping
track of the experiment models.

In [17]:
ues.metamodel.to_yaml_file(f"{DIR}/metamodel.ues.yaml")
ues.deployment_model.to_yaml_file(f"{DIR}/deployment_model.ues.yaml")